# A/B: base Whisper vs Darija-tuned model (Kaggle T4 GPU)

Decides whether [`anaszil/whisper-large-v3-turbo-darija`](https://huggingface.co/anaszil/whisper-large-v3-turbo-darija)
(a LoRA adapter on `whisper-large-v3-turbo`, Darija WER ~24.9%) is worth integrating.

It runs **both** models on the **Darija chunks** of the bundled sample and prints the
transcripts side by side:

- **BASE** = `faster-whisper large-v3`, `language=ar` — what our pipeline produces today.
- **DARIJA** = `whisper-large-v3-turbo` + the Darija LoRA adapter (HF transformers).

If DARIJA is clearly more faithful on real broadcast Darija, integrating it as a
per-chunk router target (Darija→this model, French/MSA→base) is justified. If not, the
private-corpus WER didn't survive the domain gap and we keep base large-v3.

**Setup:** Accelerator → **GPU T4**, Internet **On**. Upload the `transcription/` folder
as a Kaggle Dataset (it includes `samples/jamel_debbouze_darija.mp3`).

## 1. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.0" "transformers>=4.44" "peft>=0.11" accelerate

## 2. Load our pipeline code + sample

In [ ]:
import sys, glob, os
cands = glob.glob("/kaggle/input/**/src/transcribe.py", recursive=True)
ROOT = os.path.dirname(os.path.dirname(cands[0])) if cands else "/kaggle/input/transcription"
sys.path.insert(0, os.path.join(ROOT, "src"))
import transcribe
CLIP = os.path.join(ROOT, "samples", "jamel_debbouze_darija.mp3")
assert glob.glob(CLIP), f"clip not found: {CLIP}"
print("ROOT:", ROOT, "\nCLIP:", CLIP)

## 3. Chunk the clip and pick the Darija chunks

Uses our own VAD chunking + per-chunk language detection (faster-whisper large-v3).
We only A/B the chunks detected as Arabic/Darija (`ar`).

In [ ]:
fw = transcribe.load_model("large-v3", device="cuda", compute_type="float16")
audio = transcribe.decode_audio(CLIP)
chunks = transcribe.vad_chunks(audio, max_chunk_s=25.0)

darija = []
for i, c in enumerate(chunks):
    lang = transcribe.detect_chunk_language(fw, c["audio"], allowed=("ar", "fr", "en"))
    if lang == "ar":
        darija.append((i, c))
print(f"{len(chunks)} chunks total, {len(darija)} detected Darija/Arabic")
print("darija chunk indices:", [i for i, _ in darija])

## 4. Load the Darija model (base turbo + LoRA adapter)

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel

BASE = "openai/whisper-large-v3-turbo"
ADAPTER = "anaszil/whisper-large-v3-turbo-darija"

processor = WhisperProcessor.from_pretrained(BASE)
base = WhisperForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float16).to("cuda")
darija_model = PeftModel.from_pretrained(base, ADAPTER).eval()

def darija_transcribe(chunk_audio):
    feats = processor(chunk_audio, sampling_rate=16000, return_tensors="pt").input_features
    feats = feats.to("cuda", dtype=torch.float16)
    with torch.no_grad():
        ids = darija_model.generate(feats, language="ar", task="transcribe", max_new_tokens=440)
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

def base_transcribe(chunk_audio):
    segs, _ = fw.transcribe(chunk_audio, language="ar", beam_size=5,
                            vad_filter=False, condition_on_previous_text=False)
    return " ".join(s.text.strip() for s in segs).strip()

## 5. Side-by-side comparison on the Darija chunks

In [ ]:
for idx, c in darija:
    print(f"\n===== chunk {idx}  [{c['start']:.1f}s - {c['end']:.1f}s] =====")
    print("BASE  (large-v3) :", base_transcribe(c["audio"]))
    print("DARIJA(turbo+lora):", darija_transcribe(c["audio"]))

## 6. How to read this

- Look for Darija chunks where **DARIJA** captures real words that **BASE** garbles or drops,
  and where it doesn't drift toward MSA spelling.
- Watch for the opposite: if the adapter hallucinates or is no better on this *broadcast*
  audio, the private-corpus WER (24.9%) didn't transfer — keep base large-v3.
- There is **no ground-truth transcript** for this clip, so this is a qualitative read, not
  a WER. If it looks promising, the next step is a small hand-transcribed Darija reference to
  measure WER properly before wiring it into the pipeline (LoRA-merge → CTranslate2 →
  per-chunk router).